# Topic 3 Lab — Parts A, B and C
**Explainable Student Dropout Prediction Using Early Academic Indicators**  
Woo Yong Kang | A24AI0123 | SAIA3113 AI Engineering

Pathway B prepares the UCI table; Pathway A prepares the two existing public papers. The branches stay separate. No dummy data or model API is used.

## Run
Open this notebook in Google Colab and select **Runtime → Run all**. Missing inputs are downloaded from the official UCI source and the already attributed project PDFs, then checked against the reviewed SHA-256 values. No ZIP upload or Drive mount is needed. Existing matching local inputs are reused.

The saved outputs identify the actual runtime. Local execution is not Colab platform evidence. Save the notebook after your Colab run; files under `/content` are temporary. The optional final download cell can export generated data/evidence.

Part C reads raw sources anew, runs both branches twice, verifies content hashes and mappings, compares all prepared artefacts with Part B, and tests a fresh workspace without Part B outputs. This is a same-environment repeatability check, not full cross-environment reproducibility or a model-quality result.

## A1. Requirements → data needs
| Requirement | Needed data | Selected source / architecture | Reason and constraint |
|---|---|---|---|
| FR2 | Explicit feature windows, original outcome labels | S01 UCI CSV; D1 → A7 | Separate enrolment and first-semester predictors; exclude second-semester, target and technical IDs from predictors. |
| FR5 | Dataset definitions, timing and interpretation context | S02 Realinho (2022), S03 Martins (2023); D2 → A6 | Prepare source-linked candidate excerpts. Every final guide claim still requires manual checking. |
| NFR4 | Fixed source bytes and documented preparation | All three sources; A7 and later A9 | Record source/output SHA-256, configuration and package versions. This does not yet test the model-evaluation repeatability target. |

Supporting controls: RAI1 keeps demographic audit fields separate; RAI2 permits only aggregate evidence in a public repository; RAI4 requires exact source locators and honest extraction limitations. Topic 2 E1/E8 are affected by schema and timing; E4 by source locators; E5 by retained audit data. Their model-level acceptance targets remain unevaluated.

The current GitHub requirements were checked against the saved Lab 2 package. Refer to `docs/week2_requirements.md`, `week2_evaluation_criteria.md`, `week2_architecture_v1.png`, `week2_architecture_decisions.md` and `week1_project_scope.md` in the existing repository. Week 3 expands data preparation, not the full architecture.

## A2. Initialise the workspace (run first)

In [1]:
from pathlib import Path
import sys, json, hashlib, platform, importlib.util, subprocess
from datetime import datetime, timezone

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    BASE_DIR = Path('/content/week3_project')
else:
    here = Path.cwd().resolve()
    BASE_DIR = here.parent if here.name == 'notebooks' else here
RAW_DIR = BASE_DIR / 'data/raw'
PROCESSED_DIR = BASE_DIR / 'data/processed'
DOCS_DIR, EVIDENCE_DIR = BASE_DIR/'docs', BASE_DIR/'evidence'
for p in [RAW_DIR, PROCESSED_DIR, DOCS_DIR, EVIDENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
RAW, BASE, OUT = RAW_DIR, BASE_DIR, PROCESSED_DIR

# Only install missing libraries. Versions actually used are saved, not assumed.
for package, tested_version in [('pandas', '2.2.3'), ('pypdf', '6.21.4')]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'{package}=={tested_version}'])
import pandas as pd
import pypdf
from pypdf import PdfReader

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False)+'\n', encoding='utf-8')
environment = {
    'runtime': 'Google Colab' if IN_COLAB else 'Local Python (sequential notebook-cell execution)',
    'python': platform.python_version(), 'pandas': pd.__version__, 'pypdf': pypdf.__version__,
    'executed_at_utc': datetime.now(timezone.utc).isoformat()
}
write_json(EVIDENCE_DIR/'environment.json', environment)
print(json.dumps(environment, indent=2))
print('Raw:', RAW_DIR, '\nPrepared:', PROCESSED_DIR)

{
  "runtime": "Google Colab",
  "python": "3.13.18",
  "pandas": "2.2.3",
  "pypdf": "6.21.4",
  "executed_at_utc": "2026-11-12T15:25:52.335486+00:00"
}
Raw: /content/week3_project/data/raw 
Prepared: /content/week3_project/data/processed


## A2–A3. Verify exact inputs and provenance
S01 was downloaded from the official UCI release on 12 November 2026. S02/S03 are the unchanged, attributed publisher PDFs already included in Lab 2 (downloaded 5 November 2026). All three carry CC BY 4.0 attribution. The source registry records the original URLs, authors, dates and byte fingerprints. The data is from Portugal and must not be described as UTM student data.

The source CSV uses **semicolons**, not commas. Never load the first CSV returned by a recursive search. An unchanged source checksum is a version check, not proof of data quality or permission for real-world student decisions.

In [2]:
SOURCES = {
 'S01': {'filename': 'uci_dropout.csv', 'original_filename': 'data.csv',
   'title': "Predict Students' Dropout and Academic Success",
   'authors': 'Realinho, V.; Vieira Martins, M.; Machado, J.; Baptista, L. (2021)',
   'url': 'https://doi.org/10.24432/C5MC89',
   'download_url': 'https://archive.ics.uci.edu/static/public/697/predict%2Bstudents%2Bdropout%2Band%2Bacademic%2Bsuccess.zip',
   'access_date': '2026-11-12', 'version': 'UCI release retrieved 2026-11-12; identified by SHA-256',
   'sha256': '3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae',
   'type': 'Structured tabular CSV', 'requirements': ['FR2', 'NFR4'],
   'licence': 'CC BY 4.0', 'licence_url': 'https://creativecommons.org/licenses/by/4.0/'},
 'S02': {'filename': 'realinho_2022.pdf', 'title': 'Predicting Student Dropout and Academic Success',
   'authors': 'Realinho, V.; Machado, J.; Baptista, L.; Martins, M. V. (2022)',
   'url': 'https://doi.org/10.3390/data7110146',
   'download_url': 'https://www.mdpi.com/2306-5729/7/11/146/pdf',
   'access_date': '2026-11-05', 'version': 'Data 7(11), 146; unchanged Lab 2 PDF',
   'sha256': '1be95db77df7714e2b0783811c414895150c45cee6bbd8a79d6bd61b3a2185fb',
   'type': 'Unstructured PDF text', 'requirements': ['FR5', 'NFR4'],
   'licence': 'CC BY 4.0', 'licence_url': 'https://creativecommons.org/licenses/by/4.0/'},
 'S03': {'filename': 'martins_2023.pdf', 'title': 'Multi-Class Phased Prediction of Academic Performance and Dropout in Higher Education',
   'authors': 'Martins, M. V.; Baptista, L.; Machado, J.; Realinho, V. (2023)',
   'url': 'https://doi.org/10.3390/app13084702',
   'download_url': 'https://www.mdpi.com/2076-3417/13/8/4702/pdf',
   'access_date': '2026-11-05', 'version': 'Applied Sciences 13(8), 4702; unchanged Lab 2 PDF',
   'sha256': '2f092cb9803c2ae1f3a9198f9fe3752c3f664c0f5307b2177417a8c057d4ab46',
   'type': 'Unstructured PDF text', 'requirements': ['FR5', 'NFR4'],
   'licence': 'CC BY 4.0', 'licence_url': 'https://creativecommons.org/licenses/by/4.0/'}
}
required = {s['filename'] for s in SOURCES.values()}
# Public-source bootstrap; existing bytes remain pinned by the checks below.
import urllib.request, io
from zipfile import ZipFile
for sid, source in SOURCES.items():
    target = RAW_DIR/source['filename']
    if target.is_file():
        continue
    if sid == 'S01':
        payload = urllib.request.urlopen(source['download_url'], timeout=60).read()
        with ZipFile(io.BytesIO(payload)) as archive:
            content = archive.read('data.csv')
    else:
        url = 'https://raw.githubusercontent.com/ykwooga-afk/ai-engineering-labs/main/data/'+source['filename']
        content = urllib.request.urlopen(url, timeout=60).read()
    assert hashlib.sha256(content).hexdigest() == source['sha256'], 'Downloaded source version differs'
    target.write_bytes(content)
    print('Downloaded pinned public source:', target.name)

for sid, source in SOURCES.items():
    f = RAW_DIR/source['filename']
    assert f.is_file(), f'Missing approved input: {f.name}'
    assert sha256(f) == source['sha256'], f'Source version differs: {f.name}; review before changing the registry.'
    print(sid, f.name, f.stat().st_size, 'bytes; SHA-256 MATCH')
write_json(BASE_DIR/'data/source_registry.json', SOURCES)

Downloaded pinned public source: uci_dropout.csv
Downloaded pinned public source: realinho_2022.pdf
Downloaded pinned public source: martins_2023.pdf
S01 uci_dropout.csv 533230 bytes; SHA-256 MATCH
S02 realinho_2022.pdf 6572720 bytes; SHA-256 MATCH
S03 martins_2023.pdf 859211 bytes; SHA-256 MATCH


## A3. Inspect the loaded table and two representative subsets
Two disjoint, source-order subsets (first and last 100 records) provide ingestion evidence without publishing individual student profiles. They are **inspection subsets, not train/test partitions**. Full-cohort class counts describe the released benchmark and are not used to optimise a model. No fitted transforms or model selection occur here.

In [3]:
raw_df = pd.read_csv(RAW_DIR/'uci_dropout.csv', sep=';')
assert raw_df.shape == (4424, 37), 'Unexpected reviewed release shape'
assert set(raw_df['Target']) == {'Dropout', 'Enrolled', 'Graduate'}
inspection = {
    'rows': len(raw_df), 'columns': len(raw_df.columns),
    'class_counts': {str(k): int(v) for k,v in raw_df['Target'].value_counts().items()},
    'missing_cells': int(raw_df.isna().sum().sum()),
    'exact_duplicate_rows': int(raw_df.duplicated().sum()),
    'dtypes': {str(k): str(v) for k,v in raw_df.dtypes.items()},
    'column_names_with_surrounding_whitespace': [repr(c) for c in raw_df.columns if c != c.strip()],
    'subsets': []
}
for label, sample in [('first_100_source_rows', raw_df.iloc[:100]), ('last_100_source_rows', raw_df.iloc[-100:])]:
    observation = {'subset': label, 'rows': len(sample), 'columns': sample.shape[1],
      'missing_cells': int(sample.isna().sum().sum()),
      'class_counts': {str(k): int(v) for k,v in sample['Target'].value_counts().items()},
      'admission_grade_min': float(sample['Admission grade'].min()),
      'admission_grade_max': float(sample['Admission grade'].max())}
    inspection['subsets'].append(observation)
print(json.dumps(inspection, indent=2))
write_json(EVIDENCE_DIR/'table_inspection.json', inspection)

{
  "rows": 4424,
  "columns": 37,
  "class_counts": {
    "Graduate": 2209,
    "Dropout": 1421,
    "Enrolled": 794
  },
  "missing_cells": 0,
  "exact_duplicate_rows": 0,
  "dtypes": {
    "Marital status": "int64",
    "Application mode": "int64",
    "Application order": "int64",
    "Course": "int64",
    "Daytime/evening attendance\t": "int64",
    "Previous qualification": "int64",
    "Previous qualification (grade)": "float64",
    "Nacionality": "int64",
    "Mother's qualification": "int64",
    "Father's qualification": "int64",
    "Mother's occupation": "int64",
    "Father's occupation": "int64",
    "Admission grade": "float64",
    "Displaced": "int64",
    "Educational special needs": "int64",
    "Debtor": "int64",
    "Tuition fees up to date": "int64",
    "Gender": "int64",
    "Scholarship holder": "int64",
    "Age at enrollment": "int64",
    "International": "int64",
    "Curricular units 1st sem (credited)": "int64",
    "Curricular units 1st sem (enrolled)"

## A3. Inspect both PDF sources and retain page identity

In [4]:
pdf_pages, pdf_inspection = {}, []
for sid in ['S02', 'S03']:
    source = SOURCES[sid]
    reader = PdfReader(RAW_DIR/source['filename'])
    pages = [page.extract_text() or '' for page in reader.pages]
    assert pages and all(text.strip() for text in pages), 'Empty extracted page: inspect OCR requirements.'
    pdf_pages[sid] = pages
    info = {'source_id': sid, 'pages': len(pages), 'extracted_characters': sum(map(len,pages)),
            'page_1_sample': pages[0][:350], 'page_5_sample': pages[4][:350],
            'pages_over_10000_characters': [i+1 for i,t in enumerate(pages) if len(t)>10000],
            'pages_with_peer_review_text': [i+1 for i,t in enumerate(pages) if 'FOR PEER REVIEW' in t],
            'licence_link_in_pdf': any('creativecommons.org' in t for t in pages)}
    pdf_inspection.append(info)
    print(json.dumps(info, indent=2, ensure_ascii=False))
write_json(EVIDENCE_DIR/'pdf_inspection.json', pdf_inspection)

{
  "source_id": "S02",
  "pages": 17,
  "extracted_characters": 99741,
  "page_1_sample": "Citation: Realinho, V .; Machado, J.;\nBaptista, L.; Martins, M.V . Predicting\nStudent Dropout and Academic\nSuccess. Data 2022, 7, 146.\nhttps://doi.org/10.3390/\ndata7110146\nAcademic Editors: Antonio\nSarasa Cabezuelo and Ramón\nGonzález del Campo\nRodríguez Barbero\nReceived: 11 October 2022\nAccepted: 25 October 2022\nPublished: 28 October 2022\nPublisher",
  "page_5_sample": "Data 2022, 7, 146 5 of 17\nattributes and mean for numeric attributes), the median of each attribute value, the disper-\nsion of the attribute values (the entropy of the value distribution for categorical attributes\nand coefﬁcient of variation for numeric attributes), and the minimum and maximum value\nfor numerical attributes only.\nTable 2. Basic sta",
  "pages_over_10000_characters": [
    5,
    6
  ],
  "pages_with_peer_review_text": [
    4,
    5,
    6,
    7,
    8,
    9,
    11
  ],
  "licence_link_in_pdf

**Observed limitations.** S02 contains text extraction artefacts, including repeated/overlapping table text and peer-review header text. Visual inspection of PDF page 5 confirms that the rendered table is much cleaner than its extracted text. S03 page 5 has a readable feature-timing table, but plain extraction does not preserve its cell relationships. The first page of each PDF also contains a publisher sidebar. We retain the originals and page locators; no automatic table reconstruction, uncertain deduplication or OCR claim is made.

The 2023 paper describes **4,433** study records in section 4.1 (PDF p.4), whereas the downloaded UCI release has **4,424**. The papers supply context, not a second copy of the benchmark. Counts and columns are taken from the actual CSV; differences between study versions remain explicit.

## A4. Data inventory
The final cell writes `docs/week3_data_inventory.md` from the observed counts and registry, including rights, raw/prepared locations, limitations and requirement links.

## B1. Select relevant preparation pathways
**B (tabular)** supports D1/A7 and FR2. **A (documents)** supports D2/A6 and FR5. Images, JSON/API, audio and time-series processing are not required. CSV-to-JSON metadata does not make this a JSON-source project.

## B2–B3. Tabular comparison and preparation
Compare **A: unchanged parsed input** with **B: trim column/text whitespace and quarantine duplicate records**. Preserve the original spelling `Nacionality` with a documented meaning; only accidental surrounding whitespace is removed. Inspect conflicting duplicate feature vectors before removing any exact duplicate. There is no statistical imputation, scaling, encoding, outlier deletion, balancing or train/test split in this lab.

In [5]:
renames = {c: c.strip() for c in raw_df.columns if c != c.strip()}
clean_df = raw_df.rename(columns=renames).copy()
assert clean_df.columns.is_unique, 'Column trimming creates a name collision'
text_cells_changed = 0
for c in clean_df.select_dtypes(include=['object', 'string']).columns:
    old = clean_df[c].copy()
    clean_df[c] = clean_df[c].astype('string').str.strip()
    text_cells_changed += int((old.fillna('') != clean_df[c].fillna('')).sum())
features = [c for c in clean_df if c != 'Target']
label_counts_per_vector = clean_df.groupby(features, dropna=False)['Target'].transform('nunique')
conflict_mask = label_counts_per_vector.gt(1)
exact_mask = clean_df.duplicated(keep='first') & ~conflict_mask
quarantine_mask = conflict_mask | exact_mask
prepared = clean_df.loc[~quarantine_mask].copy()
row_ids = pd.Series([f'uci697-{i+1:06d}' for i in range(len(clean_df))], index=clean_df.index)
prepared.insert(0, 'row_id', row_ids.loc[prepared.index])
quarantine = clean_df.loc[quarantine_mask].copy()
quarantine.insert(0, 'row_id', row_ids.loc[quarantine.index])
quarantine['reason'] = ['conflicting_labels' if conflict_mask.loc[i] else 'exact_duplicate' for i in quarantine.index]

prepared_path = PROCESSED_DIR/'prepared_table.csv'
prepared.to_csv(prepared_path, index=False, lineterminator='\n')
quarantine.to_csv(PROCESSED_DIR/'quarantined_rows.csv', index=False, lineterminator='\n')
comparison = {
    'baseline': {'rows': len(raw_df), 'source_columns': raw_df.shape[1],
                 'whitespace_headers': len(renames), 'missing_cells': int(raw_df.isna().sum().sum())},
    'selected': {'rows': len(prepared), 'columns_including_row_id': prepared.shape[1],
                 'renames': {repr(k): v for k,v in renames.items()},
                 'text_cells_changed': text_cells_changed,
                 'exact_duplicates_quarantined': int(exact_mask.sum()),
                 'conflicting_rows_quarantined': int(conflict_mask.sum()),
                 'missing_cells': int(prepared.isna().sum().sum())},
    'decision': 'Trim whitespace; quarantine conflicts and repeated exact rows; preserve all other source values.',
    'benefit': 'Stable schema and auditable record handling without learned preprocessing.',
    'limitation': 'No observed missing/duplicate rows in this release; does not prove future-source quality or model fitness.'
}
print(json.dumps(comparison, indent=2))
write_json(EVIDENCE_DIR/'table_comparison.json', comparison)

{
  "baseline": {
    "rows": 4424,
    "source_columns": 37,
    "whitespace_headers": 1,
    "missing_cells": 0
  },
  "selected": {
    "rows": 4424,
    "columns_including_row_id": 38,
    "renames": {
      "'Daytime/evening attendance\\t'": "Daytime/evening attendance"
    },
    "text_cells_changed": 0,
    "exact_duplicates_quarantined": 0,
    "conflicting_rows_quarantined": 0,
    "missing_cells": 0
  },
  "decision": "Trim whitespace; quarantine conflicts and repeated exact rows; preserve all other source values.",
  "benefit": "Stable schema and auditable record handling without learned preprocessing.",
  "limitation": "No observed missing/duplicate rows in this release; does not prove future-source quality or model fitness."
}


### B2. Separate predictor windows, labels and audit fields
The complete prepared table is an **audit/staging table**, not a model input. A conservative V1 allowlist retains eight clearly named admission/pre-entry fields for T0 and adds six first-semester indicators for T1. Other potentially eligible socioeconomic fields are not selected in this small V1; exclusion does not assert that the original study lacked them.

Gender and `Nacionality` are audit-only. Age is allowed as both an enrolment predictor and audit variable, as specified in Lab 2. Debtor, fee and scholarship fields remain excluded under the existing specification even though the 2023 paper groups them with base information. Annual macroeconomic values lack a verified within-year release-time cutoff, so this version conservatively excludes them. Feature policy evidence: UCI documentation; S03 §4.2, Tables 1–2 (PDF p.5); S02 feature descriptions. These sources do not establish every row's real-time snapshot validity.

In [6]:
T0 = ['Application mode', 'Application order', 'Course', 'Daytime/evening attendance',
      'Previous qualification', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment']
SEM1 = [c for c in clean_df if c.startswith('Curricular units 1st sem')]
SEM2 = [c for c in clean_df if c.startswith('Curricular units 2nd sem')]
T1 = T0 + SEM1
AUDIT = ['Gender', 'Nacionality', 'Age at enrollment']
UNCERTAIN = ['Debtor', 'Tuition fees up to date', 'Scholarship holder', 'Unemployment rate', 'Inflation rate', 'GDP']
assert len(SEM1) == 6 and len(SEM2) == 6
policy = []
for c in clean_df:
    if c == 'Target': role, reason = 'label_only', 'Outcome at normal course duration, not a predictor.'
    elif c in SEM2: role, reason = 'excluded_future', 'Second-semester information is later than both cutoffs.'
    elif c in ['Gender','Nacionality']: role, reason = 'audit_only', 'FR2 / ADR5 excludes these fields from V1 predictors.'
    elif c in UNCERTAIN: role, reason = 'excluded_timing', 'Precise snapshot or release-time cutoff not verified; retain existing conservative exclusion.'
    elif c in SEM1: role, reason = 'T1_only', 'Semester-one aggregate; retrospective end-of-semester comparison only.'
    elif c in T0: role, reason = 'T0_and_T1', 'Admission/pre-entry field or age at enrolment; conservative V1 selection.'
    else: role, reason = 'not_selected_v1', 'Retained in staging; outside the conservative eight-field T0 allowlist.'
    policy.append({'column': c, 'role': role, 'T0_predictor': c in T0, 'T1_predictor': c in T1,
      'audit_field': c in AUDIT, 'reason': reason,
      'evidence': 'Week 2 FR2/ADR1/ADR5; UCI C5MC89 documentation; Martins 2023 section 4.2, PDF p.5'})
pd.DataFrame(policy).to_csv(DOCS_DIR/'week3_feature_policy.csv', index=False)
for filename, columns in [('features_t0.csv', T0), ('features_t1.csv', T1),
                           ('targets.csv', ['Target']), ('audit_attributes.csv', AUDIT)]:
    prepared[['row_id']+columns].to_csv(PROCESSED_DIR/filename, index=False, lineterminator='\n')
write_json(PROCESSED_DIR/'feature_allowlists.json', {'version': 'week3-ab-v1', 'T0': T0, 'T1': T1,
   'join_key_not_predictor': 'row_id', 'target_file': 'targets.csv', 'audit_file': 'audit_attributes.csv'})
print('T0 predictors:', len(T0), '\nT1 predictors:', len(T1))
print(pd.DataFrame(policy).groupby('role').size().to_string())
print('Raw/prepared rows are private working artefacts; public evidence contains aggregates only.')

T0 predictors: 8 
T1 predictors: 14
role
T0_and_T1          8
T1_only            6
audit_only         2
excluded_future    6
excluded_timing    6
label_only         1
not_selected_v1    8
Raw/prepared rows are private working artefacts; public evidence contains aggregates only.


### B3. Validate tabular content, not just file existence
Read the exports back and compare values, column order, row alignment, labels and forbidden predictors. Range checks are basic guards, not a full Topic 4 quality assessment. No real student row is printed. A synthetic example is unnecessary because the two aggregate subsets already demonstrate ingestion.

In [7]:
checks = []
def check(name, condition):
    assert bool(condition), name
    checks.append({'check': name, 'result': 'PASS'})
exported = pd.read_csv(prepared_path)
check('Prepared table exists and is nonempty', prepared_path.is_file() and prepared_path.stat().st_size>0)
pd.testing.assert_frame_equal(exported, prepared.reset_index(drop=True), check_dtype=False)
check('Table content round-trip matches', True)
check('Row accounting', len(prepared)+len(quarantine)==len(raw_df))
check('Unique technical row keys', prepared['row_id'].is_unique)
check('Three original target classes retained', set(prepared['Target'])==set(raw_df['Target']))
check('Admission grades in documented 0..200 range', prepared['Admission grade'].between(0,200).all())
check('Previous grades in documented 0..200 range', prepared['Previous qualification (grade)'].between(0,200).all())
check('No negative semester-one counts', (prepared[[c for c in SEM1 if '(grade)' not in c]]>=0).all().all())
check('Semester-one grade in 0..20 range', prepared['Curricular units 1st sem (grade)'].between(0,20).all())
for timepoint, allowed in [('t0',T0),('t1',T1)]:
    view = pd.read_csv(PROCESSED_DIR/f'features_{timepoint}.csv')
    check(f'{timepoint}: exact allowlist order', list(view.columns)==['row_id']+allowed)
    check(f'{timepoint}: row alignment', view['row_id'].tolist()==prepared['row_id'].tolist())
    check(f'{timepoint}: forbidden predictors absent', not (set(allowed)&set(SEM2+['Target','row_id','Gender','Nacionality']+UNCERTAIN)))
    pd.testing.assert_frame_equal(view, prepared[['row_id']+allowed].reset_index(drop=True), check_dtype=False)
    check(f'{timepoint}: exact selected values retained', True)
for filename, columns in [('targets.csv',['Target']),('audit_attributes.csv',AUDIT)]:
    actual = pd.read_csv(PROCESSED_DIR/filename)
    pd.testing.assert_frame_equal(actual, prepared[['row_id']+columns].reset_index(drop=True), check_dtype=False)
    check(filename+': content and alignment', True)
print(pd.DataFrame(checks).to_string(index=False))

                                      check result
      Prepared table exists and is nonempty   PASS
           Table content round-trip matches   PASS
                             Row accounting   PASS
                  Unique technical row keys   PASS
     Three original target classes retained   PASS
Admission grades in documented 0..200 range   PASS
 Previous grades in documented 0..200 range   PASS
            No negative semester-one counts   PASS
          Semester-one grade in 0..20 range   PASS
                  t0: exact allowlist order   PASS
                          t0: row alignment   PASS
            t0: forbidden predictors absent   PASS
         t0: exact selected values retained   PASS
                  t1: exact allowlist order   PASS
                          t1: row alignment   PASS
            t1: forbidden predictors absent   PASS
         t1: exact selected values retained   PASS
         targets.csv: content and alignment   PASS
audit_attributes.csv: content a

## B2–B3. Document preparation comparison
Both alternatives use the **same minimally cleaned page text** and a 1,000-character cap, with zero overlap.

- **Baseline:** fixed character slices across the concatenated document. It is simple, but may cut across page boundaries and does not retain an exact page locator.
- **Selected:** page-aware paragraph/line-boundary packing. Prefer paragraph breaks, then line breaks, then whitespace when a block exceeds the cap. Never cross a PDF page. This is a small structure-aware heuristic, **not semantic chunking or a full recursive library splitter**.

Cleaning removes exact running headers and collapses horizontal whitespace. Common typographic ligatures are expanded, and Unicode is normalised. The original PDFs are unchanged. We do not guess dehyphenation, delete repeated substantive passages, repair tables or remove the first-page sidebar. Every chunk is marked for manual review before FR5 guide use.

In [8]:
import re, unicodedata
SIZE = 1000
def clean_page(text):
    text = unicodedata.normalize('NFC', text)
    for old,new in {'ﬀ':'ff','ﬁ':'fi','ﬂ':'fl','ﬃ':'ffi','ﬄ':'ffl'}.items():
        text = text.replace(old,new)
    text = text.replace('\r\n','\n').replace('\r','\n')
    lines = []
    for line in text.split('\n'):
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if re.fullmatch(r'(Data 2022, 7, (?:146|x FOR PEER REVIEW)|Appl\. Sci\. 2023, 13, 4702)\s+\d+ of \d+', line):
            continue
        lines.append(line)
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

def page_spans(text, size=1000):
    start = 0
    while start < len(text):
        while start < len(text) and text[start].isspace(): start += 1
        if start == len(text): break
        end = min(start+size, len(text))
        if end < len(text):
            lower = start + size//2
            for marker in ['\n\n', '\n', ' ']:
                pos = text.rfind(marker, lower, end)
                if pos >= lower:
                    end = pos+len(marker)
                    break
        stop = end
        while stop > start and text[stop-1].isspace(): stop -= 1
        yield start, stop
        start = end

clean_pages, doc_comparison, chunks = {}, [], []
for sid in ['S02','S03']:
    pages = [clean_page(text) for text in pdf_pages[sid]]
    clean_pages[sid] = pages
    combined = '\n\n'.join(pages)
    fixed = [combined[i:i+SIZE] for i in range(0,len(combined),SIZE)]
    page_boundaries, pos = [], 0
    for page in pages[:-1]:
        pos += len(page)
        page_boundaries.append(pos)
        pos += 2
    source_chunks = []
    for page_number, text in enumerate(pages,1):
        flags = ['manual_source_check_required']
        raw = pdf_pages[sid][page_number-1]
        if len(raw)>10000: flags.append('unusually_dense_extraction')
        if 'FOR PEER REVIEW' in raw: flags.append('peer_review_text_in_extraction')
        if 'Table ' in text: flags.append('possible_table_reading_order_loss')
        if page_number==1: flags.append('publisher_sidebar_retained')
        for start,end in page_spans(text,SIZE):
            chunk = {'record_id': f"{sid}-{SOURCES[sid]['sha256'][:12]}-p{page_number:03d}-c{start:06d}",
              'text': text[start:end], 'source_id': sid, 'source': 'data/raw/'+SOURCES[sid]['filename'],
              'source_sha256': SOURCES[sid]['sha256'], 'source_url': SOURCES[sid]['url'],
              'source_version': SOURCES[sid]['version'], 'page': page_number,
              'start_char_in_clean_page': start, 'end_char_in_clean_page': end,
              'processing': 'page-aware paragraph/line-boundary packing; max_chars=1000; overlap=0; minimal-clean-v1',
              'licence': 'CC BY 4.0', 'status': 'candidate_for_manual_review', 'warnings': flags}
            source_chunks.append(chunk)
    chunks.extend(source_chunks)
    doc_comparison.append({'source_id':sid, 'pages':len(pages),
      'raw_characters':sum(map(len,pdf_pages[sid])), 'clean_characters':sum(map(len,pages)),
      'fixed_chunks':len(fixed), 'selected_chunks':len(source_chunks),
      'fixed_chunks_crossing_page_boundary':sum(any(i<b<min(i+SIZE,len(combined)) for b in page_boundaries) for i in range(0,len(combined),SIZE)),
      'selected_chunks_crossing_page_boundary':0,
      'fixed_first_sample':fixed[0][:160], 'selected_first_sample':source_chunks[0]['text'][:160]})

text_path = PROCESSED_DIR/'prepared_text.jsonl'
with text_path.open('w',encoding='utf-8') as handle:
    for row in chunks: handle.write(json.dumps(row,ensure_ascii=False)+'\n')
write_json(EVIDENCE_DIR/'document_comparison.json', doc_comparison)
print(json.dumps(doc_comparison,indent=2,ensure_ascii=False))
print('Selected benefit: exact PDF page provenance and bounded chunks.')
print('Accepted limitation: page breaks can split a logical argument; line boundaries do not guarantee semantic completeness.')

[
  {
    "source_id": "S02",
    "pages": 17,
    "raw_characters": 99741,
    "clean_characters": 95758,
    "fixed_chunks": 96,
    "selected_chunks": 108,
    "fixed_chunks_crossing_page_boundary": 16,
    "selected_chunks_crossing_page_boundary": 0,
    "fixed_first_sample": "Citation: Realinho, V .; Machado, J.;\nBaptista, L.; Martins, M.V . Predicting\nStudent Dropout and Academic\nSuccess. Data 2022, 7, 146.\nhttps://doi.org/10.3390/\n",
    "selected_first_sample": "Citation: Realinho, V .; Machado, J.;\nBaptista, L.; Martins, M.V . Predicting\nStudent Dropout and Academic\nSuccess. Data 2022, 7, 146.\nhttps://doi.org/10.3390/\n"
  },
  {
    "source_id": "S03",
    "pages": 15,
    "raw_characters": 50591,
    "clean_characters": 50216,
    "fixed_chunks": 51,
    "selected_chunks": 60,
    "fixed_chunks_crossing_page_boundary": 14,
    "selected_chunks_crossing_page_boundary": 0,
    "fixed_first_sample": "Citation: Martins, M.V .; Baptista, L.;\nMachado, J.; Realinho, V . M

## B3. Validate document exports and source coverage

In [9]:
loaded_chunks = [json.loads(line) for line in text_path.read_text(encoding='utf-8').splitlines() if line.strip()]
check('Document export exact JSONL round-trip', loaded_chunks==chunks)
check('Document record IDs unique', len({r['record_id'] for r in chunks})==len(chunks))
check('Every chunk nonempty and bounded', all(0<len(r['text'].strip())<=SIZE for r in chunks))
check('Both document sources represented', {r['source_id'] for r in chunks}=={'S02','S03'})
for sid in ['S02','S03']:
    for page_number,text in enumerate(clean_pages[sid],1):
        selected = [r for r in chunks if r['source_id']==sid and r['page']==page_number]
        assert selected, f'Missing page {sid}:{page_number}'
        for row in selected:
            assert row['text']==text[row['start_char_in_clean_page']:row['end_char_in_clean_page']]
        # Only whitespace between spans may be omitted; substantive content must survive.
        normal = lambda value: re.sub(r'\s+', '', value)
        assert normal(''.join(r['text'] for r in selected))==normal(text)
    check(sid+': all pages covered with exact source spans', True)
check('Source hashes match for all document records', all(r['source_sha256']==sha256(RAW_DIR/SOURCES[r['source_id']]['filename']) for r in chunks))
print('Document validation: PASS;',len(chunks),'chunks across',sum(len(v) for v in clean_pages.values()),'pages')
print('Representative prepared record:')
sample = next(r for r in chunks if r['source_id']=='S03' and r['page']==5)
print(json.dumps({k:(v[:220]+'...' if k=='text' else v) for k,v in sample.items()},indent=2,ensure_ascii=False))

Document validation: PASS; 168 chunks across 32 pages
Representative prepared record:
{
  "record_id": "S03-2f092cb9803c-p005-c000000",
  "text": "have in common the demographic, social- and macro- economic information but differ in\nthe content of the academic information: a base dataset (S0), with academic information\nrestricted to the students’ path at the time o...",
  "source_id": "S03",
  "source": "data/raw/martins_2023.pdf",
  "source_sha256": "2f092cb9803c2ae1f3a9198f9fe3752c3f664c0f5307b2177417a8c057d4ab46",
  "source_url": "https://doi.org/10.3390/app13084702",
  "source_version": "Applied Sciences 13(8), 4702; unchanged Lab 2 PDF",
  "page": 5,
  "start_char_in_clean_page": 0,
  "end_char_in_clean_page": 983,
  "processing": "page-aware paragraph/line-boundary packing; max_chars=1000; overlap=0; minimal-clean-v1",
  "licence": "CC BY 4.0",
  "status": "candidate_for_manual_review",
  "warnings": [
    "manual_source_check_required",
    "possible_table_reading_order_loss"


## B4. Register lineage, generate the common manifest, and verify it
The registry maps each actual source to every derived data artefact. The combined document JSONL has **two source-to-output entries**, one for each contributing PDF. `week3_manifest.json` is the Part B preparation manifest; it is not a Part C pipeline-run manifest.

Row IDs encode source-order positions under the pinned CSV checksum. They are internal join keys, never predictors or public student identifiers. PDF chunk offsets refer to the cleaned page text, with the unchanged PDF hash and page retained for verification.

In [10]:
lineage_entries = []
def register_lineage(sid, output, processing, **details):
    src = RAW_DIR/SOURCES[sid]['filename']
    output = Path(output)
    assert src.is_file() and output.is_file() and output.stat().st_size>0
    lineage_entries.append({'source_id': sid,
        'source_filename': str(src.relative_to(BASE_DIR)), 'source_sha256':sha256(src),
        'prepared_output':str(output.relative_to(BASE_DIR)), 'prepared_sha256':sha256(output),
        'source_version':SOURCES[sid]['version'], 'source_url':SOURCES[sid]['url'],
        'access_rights':SOURCES[sid]['licence'], 'attribution':SOURCES[sid]['authors'],
        'processing_note':processing, 'status':'Prepared; full quality review pending', **details})
for name in ['prepared_table.csv','quarantined_rows.csv','features_t0.csv','features_t1.csv','targets.csv','audit_attributes.csv','feature_allowlists.json']:
    register_lineage('S01',PROCESSED_DIR/name,
       'semicolon CSV load; trim header/text whitespace; quarantine conflicting labels and repeated exact rows; conservative feature roles; no fitted transforms',
       source_rows=len(raw_df), prepared_rows=None if name.endswith('.json') else (len(quarantine) if name=='quarantined_rows.csv' else len(prepared)),
       row_mapping='uci697-{one-based original data-row ordinal:06d}; keyed to pinned source hash',
       publication='Local/private working artefact; do not publish row-level CSVs under RAI2')
register_lineage('S01',DOCS_DIR/'week3_feature_policy.csv','Column role policy derived from S01 schema and reviewed Week 2 requirements',
                supporting_sources=['S03: PDF p.5, section 4.2, Tables 1-2', 'S02: feature descriptions'],
                requirement_ids=['FR2','RAI1','RAI2','NFR4'])
for sid in ['S02','S03']:
    register_lineage(sid,text_path,'pypdf extraction; minimal-clean-v1; page-aware paragraph/line-boundary packing; max_chars=1000; overlap=0',
        record_selector={'source_id':sid}, pages=len(pdf_pages[sid]),
        output_records=sum(r['source_id']==sid for r in chunks),
        provenance_limitations='Candidate text only; tables, sidebar and extraction artefacts require manual verification.')
manifest = {'pipeline_version':'week3-ab-v1', 'scope':'Part B preparation only',
    'generated_at_utc':datetime.now(timezone.utc).isoformat(), 'environment':environment,
    'records':lineage_entries}
manifest_path=BASE_DIR/'data/week3_manifest.json'
write_json(manifest_path,manifest)
saved=json.loads(manifest_path.read_text())
check('Manifest round-trip and registration count',saved==manifest and len(saved['records'])==len(lineage_entries))
for row in saved['records']:
    check(row['source_id']+' → '+row['prepared_output']+': checksums',
          sha256(BASE_DIR/row['source_filename'])==row['source_sha256'] and
          sha256(BASE_DIR/row['prepared_output'])==row['prepared_sha256'])
validation={'scope':'Parts A/B content and provenance checks; not full model evaluation',
    'checks':checks, 'passed':len(checks), 'failed':0,
    'table_rows':len(prepared), 'quarantined_rows':len(quarantine),
    'text_chunks':len(chunks), 'source_count':len(SOURCES), 'lineage_entries':len(lineage_entries),
    'quality_caveats':['PDF reading order and repeated table extraction remain unresolved.',
                      'No model training, fairness result or performance claim.',
                      'No Google Colab execution claimed unless runtime above says Google Colab.']}
write_json(EVIDENCE_DIR/'validation_report.json',validation)
print('COMMON LINEAGE VALIDATION: PASS')
print('Sources:',len(SOURCES),'| Lineage mappings:',len(lineage_entries),'| Checks passed:',len(checks))

COMMON LINEAGE VALIDATION: PASS
Sources: 3 | Lineage mappings: 10 | Checks passed: 37


## Save the Part A inventory and Part B decision evidence

In [11]:
inventory = [
 '# Week 3 Data Inventory — Part A',
 '', '**Project:** Explainable Student Dropout Prediction Using Early Academic Indicators',
 '**Student:** Woo Yong Kang | A24AI0123 | SAIA3113 AI Engineering',
 '', '## A1. Requirement and architecture traceability',
 '', '| Requirement | Needed information | Source | Architecture / evaluation link |',
 '|---|---|---|---|',
 '| FR2 | Timepoint-specific tabular features and original three-class outcome | S01 | D1 → A7; E1 timing controls |',
 '| FR5 | Source-linked definitions, timing and limits | S02, S03 | D2 → A6; E4 source support |',
 '| NFR4 | Fixed sources and explicit preparation | All | A7/A9; E8 later repeatability |',
 '', 'RAI1 separates audit attributes, RAI2 restricts public row-level output, and RAI4 requires manual guide-claim verification. Model-level acceptance targets remain unevaluated.',
 '', '## A2–A4. Source inventory',
 '', '| ID | File / canonical source | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |',
 '|---|---|---|---|---|---|'
]
for sid,s in SOURCES.items():
    observed = (f"{len(raw_df):,} rows × {raw_df.shape[1]} columns; {inspection['missing_cells']} missing cells; {inspection['exact_duplicate_rows']} exact duplicates; two 100-row subsets inspected"
                if sid=='S01' else f"{len(pdf_pages[sid])} pages; {sum(map(len,pdf_pages[sid])):,} extracted characters; pp.1 and 5 sampled; all pages nonempty")
    purpose = 'later tabular prediction; not live student assessment' if sid=='S01' else 'candidate reference-guide evidence, not training labels'
    inventory.append(f"| {sid} | `data/raw/{s['filename']}`; [{s['title']}]({s['url']}) | {s['type']}; {purpose} | {s['version']}; accessed {s['access_date']}; {s['authors']} | {s['licence']} with attribution; no institution login needed | {observed} |")
inventory += ['', '## Observable ingestion evidence', '',
    'The CSV was explicitly parsed with `sep=";"`. `Daytime/evening attendance` has a trailing tab in the raw header. The published spelling `Nacionality` is retained and documented as nationality.',
    '', '| Source-order inspection subset | Rows | Missing cells | Class counts | Admission grade min / max |',
    '|---|---:|---:|---|---|']
for sample in inspection['subsets']:
    inventory.append(f"| {sample['subset']} | {sample['rows']} | {sample['missing_cells']} | {sample['class_counts']} | {sample['admission_grade_min']} / {sample['admission_grade_max']} |")
inventory += ['', 'These disjoint subsets are ingestion checks, not train/test partitions. Notebook output and `evidence/table_inspection.json` retain the loaded schema, types and actual aggregate values. `evidence/pdf_inspection.json` retains text samples and page-level warnings.',
    '', '## Source identity and raw/prepared storage', '',
    'Raw originals: `data/raw/`. Part B exports: `data/processed/`. Source registry: `data/source_registry.json`. Common Part B lineage: `data/week3_manifest.json`.',
    'In Colab these paths are under `/content/week3_project`; they are temporary. Download generated outputs and save the executed notebook before disconnecting. The private submission ZIP contains the working CSVs. No Google Drive storage is used. See the project README for repository evidence.',
    '', '| Source | SHA-256 |', '|---|---|']
for sid,s in SOURCES.items(): inventory.append(f"| {sid} | `{s['sha256']}` |")
inventory += ['', '## Rights, scope and quality limitations', '',
    '- The UCI licence permits attributed reuse, but Week 2 RAI2 is stricter for publication: keep row-level source/prepared tables private and publish aggregate evidence. The included `.gitignore` supports that rule.',
    '- Copyright for both CC BY 4.0 papers remains with the authors. The original PDF bytes are unchanged; derived text carries source URL, version, page and processing description.',
    '- The sources concern Portuguese higher education. They do not establish UTM performance or support real student decisions.',
    '- The 2023 study reports 4,433 records (§4.1, PDF p.4); the actual UCI release has 4,424. Do not silently equate their versions.',
    '- S02 has duplicated/overlapping table extraction and peer-review text. S03 tables lose their layout in plain text. Both include a first-page publisher sidebar. All guide chunks remain candidates for manual checking.',
    '- Model training, embeddings, vector indexing, full Topic 4 quality/fairness assessment are not claimed. Part C follows in the same notebook.',
    '', '## A4 checkpoint', '',
    'The data sources are justified by FR2 and FR5, all three pinned files were loaded, and the notebook shows their actual structure/content. One tabular dataset plus two supporting documents is sufficient; the six lab pathways are alternatives.',
    '', '## Source references', '',
    '1. Realinho et al. (2021). UCI Predict Students’ Dropout and Academic Success. https://doi.org/10.24432/C5MC89',
    '2. Realinho et al. (2022). Data 7(11), 146. https://doi.org/10.3390/data7110146',
    '3. Martins et al. (2023). Applied Sciences 13(8), 4702. https://doi.org/10.3390/app13084702',
    '4. Existing project requirements: https://github.com/ykwooga-afk/ai-engineering-labs/blob/main/docs/week2_requirements.md',
    '5. Licence: https://creativecommons.org/licenses/by/4.0/']
(DOCS_DIR/'week3_data_inventory.md').write_text('\n'.join(inventory)+'\n',encoding='utf-8')

decisions = ['# Week 3 Preparation Decisions — Part B', '',
 '**Student:** Woo Yong Kang | A24AI0123', '',
 '## B1. Selected pathways', '',
 'Pathway B prepares the actual UCI benchmark for FR2. Pathway A prepares the two existing domain PDFs for FR5. These branches remain separate. Other modalities and model integration are outside this work.', '',
 '## B2–B3. Comparison and choice', '',
 '| Branch | Baseline | Selected preparation | Observed evidence | Benefit / accepted limitation |',
 '|---|---|---|---|---|',
 f"| Table | Parsed source without cleanup | Trim whitespace; quarantine exact repeats and conflicting labels | {len(raw_df):,} → {len(prepared):,} rows; {len(renames)} header corrected; {text_cells_changed} text cells changed; {int(quarantine_mask.sum())} rows quarantined | Stable schema; no fabricated cleaning or fitted preprocessing. This release does not exercise duplicate-removal outcomes. |"]
for item in doc_comparison:
    decisions.append(f"| {item['source_id']} PDF | Whole-document fixed 1,000-character slices | Page-aware paragraph/line packing, max 1,000 characters, no overlap | {item['fixed_chunks']} baseline vs {item['selected_chunks']} selected chunks; cross-page chunks {item['fixed_chunks_crossing_page_boundary']} → 0 | Exact page location; page breaks and line boundaries may still interrupt meaning. Table extraction remains unverified. |")
decisions += ['', 'The same cleaned content and size cap are used for both document candidates. Baseline and selected counts come from actual execution. Neither chunking approach is semantic chunking. Sample excerpts and counts are in `evidence/document_comparison.json`.',
    '', '## Predictor policy and data leakage', '',
    f'T0 has {len(T0)} conservative predictors; T1 has {len(T1)}. See `week3_feature_policy.csv` for every raw column. `feature_allowlists.json` names the exact predictor columns; `row_id` is solely a join key.',
    'All second-semester fields, target, technical ID, gender and nationality are excluded from predictor allowlists. Debtor, tuition-fee, scholarship and annual macroeconomic snapshots remain excluded pending time-cutoff verification. Other socioeconomic fields are retained in staging but are not selected in the conservative V1.',
    'The complete staging CSV intentionally retains source information for audit. Downstream training must load the explicit T0/T1 views and allowlists, never automatically use every staging column.',
    'No split, statistical imputation, scaling, encoding, balancing, model fitting or threshold optimisation is performed. Future work must use the common split and training-fold-only fitting from Week 2.',
    '', '## B3. Validation evidence', '',
    f"{len(checks)} content/provenance checks passed. Prepared rows: {len(prepared):,}; quarantined rows: {len(quarantine)}; PDF chunks: {len(chunks)}; covered PDF pages: {sum(len(v) for v in clean_pages.values())}.",
    'Checks include read-back equality, row accounting, exact predictor allowlists, target/audit alignment, basic ranges, unique chunk IDs, full cleaned-page content coverage, exact chunk offsets and source/output hashes. Header-only quarantine CSV is a valid zero-row result, not evidence that bad rows were removed.',
    'PASS means these specific checks succeeded; it is not a source-rights, fairness, model-quality or deployment certification.',
    '', '## B4. Metadata and lineage', '',
    f"The common manifest contains {len(lineage_entries)} source-to-output mappings. It records source and output SHA-256, attribution/licence, source version, processing, configuration descriptions and provenance limitations. S02 and S03 each map to their selected records within the combined `prepared_text.jsonl`.",
    'The PDF chunk records retain source URL, source hash, page, cleaned-page offsets and a stable record ID. All are marked `candidate_for_manual_review`; an excerpt is not automatically a verified reference-guide claim.',
    '', '## Saved outputs', '',
    '- `data/processed/prepared_table.csv`: complete staging table plus internal row key.',
    '- `features_t0.csv`, `features_t1.csv`: predictor views plus join key.',
    '- `targets.csv`, `audit_attributes.csv`: separate outcomes and demographic audit fields.',
    '- `quarantined_rows.csv`: rejected/repeated row log; zero rows for this release.',
    '- `feature_allowlists.json`: exact allowed predictor names.',
    '- `prepared_text.jsonl`: page-linked candidate text from both PDFs.',
    '- `data/week3_manifest.json`: common Part B lineage; not Part C run evidence.',
    '', '## Remaining work', '',
    'Run and save the notebook in Google Colab if the current execution environment is local. Part C workflow integration, diagram and decisions are supplied with the complete Week 3 package; see the project README for commit evidence. Topic 4 should review PDF extraction, feature timing, representativeness and data quality. Model-level Week 2 criteria have not been evaluated.']
(DOCS_DIR/'week3_preparation_decisions.md').write_text('\n'.join(decisions)+'\n',encoding='utf-8')
print('Saved docs/week3_data_inventory.md and docs/week3_preparation_decisions.md')
print('Actual execution environment:',environment['runtime'])

Saved docs/week3_data_inventory.md and docs/week3_preparation_decisions.md
Actual execution environment: Google Colab


# Part C — Executable V1 Data Pipeline
## C1 / Cell 7. Common pipeline functions
This cell contains the same implementation as `src/week3_pipeline.py`. Each branch runs **ingest → inspect → prepare → validate temporary outputs → store → record lineage**. Inputs and configuration are explicit arguments. No Part B dataframe, chunk list or prepared file is read by `run_pipeline`.

Validation occurs before promotion from temporary storage. The manifest is written last. A failure propagates as an exception and must not be reported as a new successful run. This is a single-process lab workflow, not a transactional distributed system; interrupted multi-file promotion would require rerunning and re-verifying the manifest.

In [12]:
"""Week 3 V1 data pipeline: ingest, inspect, prepare, validate, store, lineage.

No fitted preprocessing, split, model, retrieval index or external model API.
Both branches read raw inputs anew on each call and use explicit configuration.
"""
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import os
import platform
import re
import unicodedata
import pandas as pd
import pypdf
from pypdf import PdfReader

PIPELINE_VERSION = 'week3-v1'

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def json_write(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)+'\n', encoding='utf-8')

def clean_page(text):
    text = unicodedata.normalize('NFC', text)
    for old,new in {'ﬀ':'ff','ﬁ':'fi','ﬂ':'fl','ﬃ':'ffi','ﬄ':'ffl'}.items():
        text = text.replace(old,new)
    text = text.replace('\r\n','\n').replace('\r','\n')
    lines = []
    for line in text.split('\n'):
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if re.fullmatch(r'(Data 2022, 7, (?:146|x FOR PEER REVIEW)|Appl\. Sci\. 2023, 13, 4702)\s+\d+ of \d+', line):
            continue
        lines.append(line)
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

def page_spans(text, size=1000):
    if size <= 0:
        raise ValueError('Chunk size must be positive')
    start = 0
    while start < len(text):
        while start < len(text) and text[start].isspace(): start += 1
        if start == len(text): break
        end = min(start+size, len(text))
        if end < len(text):
            lower = start + size//2
            for marker in ['\n\n', '\n', ' ']:
                pos = text.rfind(marker, lower, end)
                if pos >= lower:
                    end = pos+len(marker)
                    break
        stop = end
        while stop > start and text[stop-1].isspace(): stop -= 1
        yield start, stop
        start = end

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def ingest(base_dir, source_ids, registry):
    """Missing files or changed source bytes stop before any output is written."""
    sources = []
    for sid in source_ids:
        source = registry[sid]
        path = Path(base_dir)/'data/raw'/source['filename']
        if not path.is_file():
            raise FileNotFoundError(f'{sid}: missing input {source["filename"]}')
        digest = file_hash(path)
        require(digest == source['sha256'], f'{sid}: source checksum mismatch; review the new version')
        sources.append({'source_id':sid, 'path':'data/raw/'+source['filename'],
            'sha256':digest, 'bytes':path.stat().st_size,
            'source_version':source['version'], 'source_url':source['url'],
            'attribution':source['authors'], 'access_rights':source['licence']})
    return sources

def prepare_tabular(base_dir, staging, cfg, registry):
    source = Path(base_dir)/'data/raw'/registry['S01']['filename']
    df = pd.read_csv(source, sep=cfg['separator'])
    require(df.shape == (cfg['expected_rows'], len(cfg['expected_raw_columns'])), 'Unexpected table dimensions')
    require(df.columns.tolist() == cfg['expected_raw_columns'], 'Unexpected table schema or column order')
    require(set(df['Target']) == set(cfg['labels']), 'Unexpected target labels')
    info = {'rows':len(df), 'columns':df.shape[1], 'missing_cells':int(df.isna().sum().sum()),
        'exact_duplicate_rows':int(df.duplicated().sum()),
        'class_counts':{str(k):int(v) for k,v in df['Target'].value_counts().items()}}
    clean = df.rename(columns=lambda s:s.strip()).copy()
    require(clean.columns.is_unique, 'Header normalization caused a collision')
    for c in clean.select_dtypes(include=['object','string']).columns:
        clean[c] = clean[c].astype('string').str.strip()
    features = [c for c in clean if c!='Target']
    conflict = clean.groupby(features,dropna=False)['Target'].transform('nunique').gt(1)
    repeated = clean.duplicated(keep='first') & ~conflict
    rejected = conflict | repeated
    keys = pd.Series([f'uci697-{i+1:06d}' for i in range(len(clean))],index=clean.index)
    prepared = clean.loc[~rejected].copy()
    prepared.insert(0,'row_id',keys.loc[prepared.index])
    quarantine = clean.loc[rejected].copy()
    quarantine.insert(0,'row_id',keys.loc[quarantine.index])
    quarantine['reason'] = ['conflicting_labels' if conflict.loc[i] else 'exact_duplicate' for i in quarantine.index]
    require(not prepared.empty,'Preparation removed all records')
    require(len(prepared)+len(quarantine)==len(df),'Row accounting failed')
    require(prepared.row_id.is_unique,'Duplicate row keys')
    require(prepared['Admission grade'].between(0,200).all(),'Admission grade outside 0..200')
    require(prepared['Previous qualification (grade)'].between(0,200).all(),'Previous grade outside 0..200')
    sem1=[c for c in clean if c.startswith('Curricular units 1st sem')]
    require((prepared[[c for c in sem1 if '(grade)' not in c]]>=0).all().all(),'Negative semester-one count')
    require(prepared['Curricular units 1st sem (grade)'].between(0,20).all(),'Semester-one grade outside 0..20')
    forbidden={c for c in clean if '2nd sem' in c} | set(cfg['excluded_timing']) | {'Target','row_id','Gender','Nacionality'}
    for timepoint in ['T0','T1']:
        allowed=cfg[timepoint]
        require(len(allowed)==len(set(allowed)),f'{timepoint}: duplicated feature')
        require(not (set(allowed)&forbidden),f'{timepoint}: prohibited predictor')
        require(set(allowed)<=set(clean.columns),f'{timepoint}: unknown feature')
    require(not any('1st sem' in c for c in cfg['T0']),'T0 contains first-semester information')
    require(cfg['T1']==cfg['T0']+sem1,'T1 does not match the selected Part B policy')
    frames={'prepared_table.csv':prepared, 'quarantined_rows.csv':quarantine,
        'features_t0.csv':prepared[['row_id']+cfg['T0']],
        'features_t1.csv':prepared[['row_id']+cfg['T1']],
        'targets.csv':prepared[['row_id','Target']],
        'audit_attributes.csv':prepared[['row_id']+cfg['audit']]}
    for name,frame in frames.items():
        frame.to_csv(staging/name,index=False,lineterminator='\n')
        actual=pd.read_csv(staging/name)
        pd.testing.assert_frame_equal(actual,frame.reset_index(drop=True),check_dtype=False)
    allowlists={'version':'week3-ab-v1','T0':cfg['T0'],'T1':cfg['T1'],
        'join_key_not_predictor':'row_id','target_file':'targets.csv','audit_file':'audit_attributes.csv'}
    json_write(staging/'feature_allowlists.json',allowlists)
    require(json.loads((staging/'feature_allowlists.json').read_text())==allowlists,'Allowlist round-trip failed')
    outputs=list(frames)+['feature_allowlists.json']
    info.update({'prepared_rows':len(prepared),'quarantined_rows':len(quarantine),
                 't0_features':len(cfg['T0']),'t1_features':len(cfg['T1'])})
    mappings=[{'source_id':'S01','source':'data/raw/'+registry['S01']['filename'],
        'output':'data/pipeline_v1/tabular/'+name,
        'row_mapping':'uci697-{one-based original data-row ordinal:06d}; fixed source hash'} for name in outputs]
    return outputs,mappings,info,[]

def prepare_documents(base_dir, staging, cfg, registry):
    require(cfg['method']=='page-aware-paragraph-line','Unsupported chunking method')
    require(cfg['overlap']==0,'V1 supports zero overlap only')
    size=cfg['chunk_size']
    require(isinstance(size,int) and size>0,'Invalid chunk size')
    chunks,info,warnings=[],[],[]
    for sid in cfg['source_ids']:
        source=registry[sid]
        reader=PdfReader(Path(base_dir)/'data/raw'/source['filename'])
        raw_pages=[p.extract_text() or '' for p in reader.pages]
        require(raw_pages and all(t.strip() for t in raw_pages),f'{sid}: empty page extraction; inspect OCR needs')
        pages=[clean_page(t) for t in raw_pages]
        count_before=len(chunks)
        for page_number,text in enumerate(pages,1):
            flags=['manual_source_check_required']
            raw=raw_pages[page_number-1]
            if len(raw)>10000:flags.append('unusually_dense_extraction')
            if 'FOR PEER REVIEW' in raw:flags.append('peer_review_text_in_extraction')
            if 'Table ' in text:flags.append('possible_table_reading_order_loss')
            if page_number==1:flags.append('publisher_sidebar_retained')
            page_chunks=[]
            for start,end in page_spans(text,size):
                row={'record_id':f"{sid}-{source['sha256'][:12]}-p{page_number:03d}-c{start:06d}",
                    'text':text[start:end],'source_id':sid,'source':'data/raw/'+source['filename'],
                    'source_sha256':source['sha256'],'source_url':source['url'],
                    'source_version':source['version'],'page':page_number,
                    'start_char_in_clean_page':start,'end_char_in_clean_page':end,
                    'processing':f'page-aware paragraph/line-boundary packing; max_chars={size}; overlap=0; minimal-clean-v1',
                    'licence':'CC BY 4.0','status':'candidate_for_manual_review','warnings':flags}
                require(0<len(row['text'].strip())<=size,f'{sid}: empty or oversized chunk')
                page_chunks.append(row)
            require(page_chunks,f'{sid}: no chunks on page {page_number}')
            normal=lambda t:re.sub(r'\s+','',t)
            require(normal(''.join(r['text'] for r in page_chunks))==normal(text),f'{sid}: page coverage loss')
            chunks.extend(page_chunks)
            if len(flags)>1:warnings.append({'source_id':sid,'page':page_number,'warnings':flags[1:]})
        info.append({'source_id':sid,'pages':len(pages),'raw_characters':sum(map(len,raw_pages)),
                     'chunks':len(chunks)-count_before})
    require(len({r['record_id'] for r in chunks})==len(chunks),'Duplicate chunk IDs')
    target=staging/'prepared_text.jsonl'
    with target.open('w',encoding='utf-8') as f:
        for row in chunks:f.write(json.dumps(row,ensure_ascii=False)+'\n')
    restored=[json.loads(line) for line in target.read_text(encoding='utf-8').splitlines() if line.strip()]
    require(restored==chunks,'Document JSONL round-trip mismatch')
    mappings=[{'source_id':sid,'source':'data/raw/'+registry[sid]['filename'],
        'output':'data/pipeline_v1/documents/prepared_text.jsonl',
        'record_selector':{'source_id':sid},'page_range':[1,len(PdfReader(Path(base_dir)/'data/raw'/registry[sid]['filename']).pages)]}
        for sid in cfg['source_ids']]
    return ['prepared_text.jsonl'],mappings,{'documents':info,'total_chunks':len(chunks)},warnings

def verify_saved_run(base_dir,record):
    """Verify hashes on disk, not just a previously printed PASS."""
    base_dir=Path(base_dir)
    for source in record['sources']:
        path=base_dir/source['path']
        require(path.is_file() and file_hash(path)==source['sha256'],'Saved run source fingerprint mismatch')
    require(record['outputs'],'No output records')
    for output in record['outputs']:
        path=base_dir/output['path']
        require(path.is_file() and path.stat().st_size>0,'Missing or empty saved output')
        require(file_hash(path)==output['sha256'] and path.stat().st_size==output['bytes'],'Saved output fingerprint mismatch')
    sources={s['source_id'] for s in record['sources']}
    paths={o['path'] for o in record['outputs']}
    require({m['source_id'] for m in record['source_output_mapping']}==sources,'Source without a lineage mapping')
    require({m['output'] for m in record['source_output_mapping']}==paths,'Output without a lineage mapping')
    for mapping in record['source_output_mapping']:
        expected=next(s['path'] for s in record['sources'] if s['source_id']==mapping['source_id'])
        require(mapping['source']==expected,'Wrong source in mapping')
    return True

def run_pipeline(kind,base_dir,registry,configuration,runtime='Local Python'):
    """Execute a complete branch from raw input without Part B in-memory data."""
    base_dir=Path(base_dir).resolve()
    require(kind in ['tabular','documents'],'Select a supported branch')
    cfg=json.loads(json.dumps(configuration))
    source_ids=['S01'] if kind=='tabular' else cfg['source_ids']
    sources=ingest(base_dir,source_ids,registry)
    out=base_dir/'data/pipeline_v1'/kind
    out.parent.mkdir(parents=True,exist_ok=True)
    with TemporaryDirectory(prefix='week3_staging_',dir=out.parent) as temporary:
        staging=Path(temporary)
        prepare=prepare_tabular if kind=='tabular' else prepare_documents
        names,mappings,inspection,warnings=prepare(base_dir,staging,cfg,registry)
        require(names and len(names)==len(set(names)),'Missing or duplicate output filenames')
        outputs=[]
        for name in names:
            path=staging/name
            require(path.is_file() and path.stat().st_size>0,'Prepared output is missing or empty')
            outputs.append({'path':str((out/name).relative_to(base_dir)),
                'sha256':file_hash(path),'bytes':path.stat().st_size})
        record={'pipeline_version':PIPELINE_VERSION,'generated_at_utc':datetime.now(timezone.utc).isoformat(),
            'data_type':kind,'runtime':runtime,
            'environment':{'python':platform.python_version(),'pandas':pd.__version__,'pypdf':pypdf.__version__},
            'configuration':cfg,'sources':sources,'inspection':inspection,'outputs':outputs,
            'source_output_mapping':mappings,'validation':{'content_roundtrip':'PASS','output_count':len(outputs)},
            'warnings':warnings,'status':'PREPARED; full quality review pending',
            'provenance_limitations':'Hashes identify bytes; they do not prove fairness, guide claim correctness or real-time feature validity.'}
        # Only validated temporary outputs are promoted; manifest is written last.
        out.mkdir(parents=True,exist_ok=True)
        for name in names:os.replace(staging/name,out/name)
        verify_saved_run(base_dir,record)
        json_write(staging/'pipeline_manifest.json',record)
        os.replace(staging/'pipeline_manifest.json',out/'pipeline_manifest.json')
    require(json.loads((out/'pipeline_manifest.json').read_text())==record,'Run manifest round-trip failed')
    print(f'SUCCESS: {kind} | Inputs: {len(sources)} | Outputs: {len(outputs)}')
    print('Manifest:',str((out/'pipeline_manifest.json').relative_to(base_dir)))
    return record

def repeatability_signature(record):
    """Execution timestamp is expected to differ; all deterministic evidence matches."""
    return {k:v for k,v in record.items() if k!='generated_at_utc'}

## C1 / Selected Cells 8B and 8A. Run both relevant branches
The schema and allowlists are the Part B decisions. The paragraph/line chunking cap remains 1,000 characters with zero overlap. Expected source identity comes from the common registry, not whichever file happens to appear first in a folder.

In [13]:
TABULAR_CONFIG = {
    'separator': ';', 'expected_rows': 4424, 'expected_raw_columns': raw_df.columns.tolist(),
    'labels': ['Dropout','Enrolled','Graduate'], 'T0': T0, 'T1': T1,
    'audit': AUDIT, 'excluded_timing': UNCERTAIN,
    'preparation': 'trim header/text whitespace; quarantine conflicting labels and repeated exact rows',
    'fitted_transforms': False
}
DOCUMENT_CONFIG = {'source_ids':['S02','S03'], 'chunk_size':1000,
    'method':'page-aware-paragraph-line', 'overlap':0, 'cleaning':'minimal-clean-v1'}
PIPELINE_CONFIG = {'tabular':TABULAR_CONFIG,'documents':DOCUMENT_CONFIG}
write_json(BASE_DIR/'data/pipeline_configuration.json',PIPELINE_CONFIG)
first_runs = {}
for kind, configuration in PIPELINE_CONFIG.items():
    first_runs[kind] = run_pipeline(kind,BASE_DIR,SOURCES,configuration,environment['runtime'])
    print('Inspection:',json.dumps(first_runs[kind]['inspection'],ensure_ascii=False))

SUCCESS: tabular | Inputs: 1 | Outputs: 7
Manifest: data/pipeline_v1/tabular/pipeline_manifest.json
Inspection: {"rows": 4424, "columns": 37, "missing_cells": 0, "exact_duplicate_rows": 0, "class_counts": {"Graduate": 2209, "Dropout": 1421, "Enrolled": 794}, "prepared_rows": 4424, "quarantined_rows": 0, "t0_features": 8, "t1_features": 14}
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Inspection: {"documents": [{"source_id": "S02", "pages": 17, "raw_characters": 99741, "chunks": 108}, {"source_id": "S03", "pages": 15, "raw_characters": 50591, "chunks": 60}], "total_chunks": 168}


## C1 / Cell 9. Repeatability, provenance and Part B parity
Run both branches again. Compare all deterministic manifest fields, including inputs, configuration, output hashes/sizes/counts, inspection and source-to-output mappings. Execution timestamps are retained as separate run evidence and excluded from equality checks. Then compare every Part C prepared output against the corresponding Part B output.

In [14]:
second_runs, repeat_results, parity_results = {}, [], []
for kind, configuration in PIPELINE_CONFIG.items():
    second = run_pipeline(kind,BASE_DIR,SOURCES,configuration,environment['runtime'])
    second_runs[kind] = second
    assert repeatability_signature(first_runs[kind]) == repeatability_signature(second), f'{kind}: repeated result changed'
    assert verify_saved_run(BASE_DIR,second)
    repeat_results.append({'pathway':kind,'result':'PASS',
        'compared':'source fingerprints, output fingerprints/sizes/counts, configuration, inspection and mappings',
        'first_generated_at_utc':first_runs[kind]['generated_at_utc'],
        'second_generated_at_utc':second['generated_at_utc']})
    for output in second['outputs']:
        baseline = PROCESSED_DIR/Path(output['path']).name
        assert baseline.is_file() and file_hash(baseline)==output['sha256'], 'Part B/C output mismatch: '+baseline.name
        parity_results.append({'pathway':kind,'output':baseline.name,'result':'PASS','sha256':output['sha256']})
    print('PASS:',kind,'— source/output fingerprints, configuration and mappings match')
write_json(EVIDENCE_DIR/'week3_run1.json',first_runs)
write_json(EVIDENCE_DIR/'week3_run2.json',second_runs)
write_json(EVIDENCE_DIR/'week3_repeatability.json',{'results':repeat_results,'part_b_parity':parity_results,
    'runtime':environment['runtime'], 'scope':'Same-environment smoke check; not model evaluation or cross-environment proof'})
print('Part B/C parity: PASS for',len(parity_results),'prepared output files')

SUCCESS: tabular | Inputs: 1 | Outputs: 7
Manifest: data/pipeline_v1/tabular/pipeline_manifest.json
PASS: tabular — source/output fingerprints, configuration and mappings match
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source/output fingerprints, configuration and mappings match
Part B/C parity: PASS for 8 prepared output files


### Independence and two failure responses
The fresh-workspace check copies **only the three raw files**, then runs the same functions. This checks that Part C does not rely on Part B's saved outputs or in-memory prepared records. Two failure exercises use temporary copies: F1 removes an input; F2 empties a saved output before provenance verification. Neither exercise alters the real project inputs or successful outputs.

In [15]:
import shutil
independence, failure_checks = [], []
with TemporaryDirectory(prefix='week3_clean_check_') as temporary:
    fresh = Path(temporary)
    (fresh/'data/raw').mkdir(parents=True)
    for source in SOURCES.values():
        shutil.copyfile(RAW_DIR/source['filename'],fresh/'data/raw'/source['filename'])
    for kind,configuration in PIPELINE_CONFIG.items():
        fresh_run = run_pipeline(kind,fresh,SOURCES,configuration,environment['runtime'])
        assert repeatability_signature(fresh_run)==repeatability_signature(second_runs[kind])
        independence.append({'pathway':kind,'result':'PASS','prepared_inputs_available':False})
    # F2: manifest verification must reject an emptied output.
    bad_output = fresh/second_runs['tabular']['outputs'][0]['path']
    bad_output.write_bytes(b'')
    try:
        verify_saved_run(fresh,second_runs['tabular'])
    except ValueError as error:
        failure_checks.append({'case':'F2 empty output','result':'PASS','response':str(error)})
    else:
        raise AssertionError('Empty output was not rejected')
    # F1: ingestion must stop when a selected source is missing.
    (fresh/'data/raw'/SOURCES['S01']['filename']).unlink()
    try:
        run_pipeline('tabular',fresh,SOURCES,TABULAR_CONFIG,environment['runtime'])
    except FileNotFoundError as error:
        failure_checks.append({'case':'F1 missing input','result':'PASS','response':str(error)})
    else:
        raise AssertionError('Missing input was not rejected')
write_json(EVIDENCE_DIR/'week3_independence_and_failures.json',{'independence':independence,'failure_checks':failure_checks})
print(json.dumps({'independence':independence,'failure_checks':failure_checks},indent=2))
print('PART C VERIFICATION: PASS')

SUCCESS: tabular | Inputs: 1 | Outputs: 7
Manifest: data/pipeline_v1/tabular/pipeline_manifest.json
SUCCESS: documents | Inputs: 2 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
{
  "independence": [
    {
      "pathway": "tabular",
      "result": "PASS",
      "prepared_inputs_available": false
    },
    {
      "pathway": "documents",
      "result": "PASS",
      "prepared_inputs_available": false
    }
  ],
  "failure_checks": [
    {
      "case": "F2 empty output",
      "result": "PASS",
      "response": "Missing or empty saved output"
    },
    {
      "case": "F1 missing input",
      "result": "PASS",
      "response": "S01: missing input uci_dropout.csv"
    }
  ]
}
PART C VERIFICATION: PASS


## C2. Executed workflow diagram
The editable Draw.io source and PNG expand only D1 → A7 data preparation and D2 → A6 document preparation from the Week 2 architecture. Model fitting, split creation and the finished guide remain later work.

![Week 3 pipeline](https://raw.githubusercontent.com/ykwooga-afk/ai-engineering-labs/main/docs/week3_pipeline_diagram.png)

[Editable Draw.io diagram](https://github.com/ykwooga-afk/ai-engineering-labs/blob/main/docs/week3_pipeline_diagram.drawio)

F1 is a missing/changed source: stop ingestion and report the source ID. F2 is a missing/empty/altered output: fail validation and withhold a new successful manifest. The executed failure exercises are recorded above.

## C3. Decisions and stage contracts
`docs/week3_pipeline_decisions.md` records requirement IDs, evidence from Part B, exact configuration, ETL/ELT interpretation, batch rationale, stage contracts, failure responses, output locations and limitations.

The design retains raw inputs first, then transforms into validated prepared outputs. It is a raw-landing/staged workflow with an ETL-style preparation subflow. Batch processing fits the three versioned sources and manual updates; a streaming system is not justified.

## C4. Saving and repository evidence
The repository README links all Week 3 artefacts. Tabular row-level CSVs stay in the private submission package or the user's local/Colab workspace under the Week 2 RAI2 publication rule. Public execution evidence contains aggregate inspection, hashes and mapping metadata. Source registry and notebook bootstrap allow the permitted UCI data to be recreated locally.

## Optional: download generated outputs in Colab
Set `DOWNLOAD_OUTPUTS = True` and run this final cell to download all currently generated data, manifests and evidence. Save the executed notebook separately in Colab. The submitted diagram/decision documents are also available in the repository.

In [16]:
DOWNLOAD_OUTPUTS = False
if IN_COLAB and DOWNLOAD_OUTPUTS:
    from zipfile import ZipFile, ZIP_DEFLATED
    target=BASE_DIR/'week3_ABC_generated_outputs.zip'
    with ZipFile(target,'w',ZIP_DEFLATED) as archive:
        for directory in [DOCS_DIR,EVIDENCE_DIR,PROCESSED_DIR,BASE_DIR/'data/pipeline_v1']:
            for path in sorted(directory.rglob('*')):
                if path.is_file():archive.write(path,path.relative_to(BASE_DIR))
        for name in ['source_registry.json','week3_manifest.json','pipeline_configuration.json']:
            path=BASE_DIR/'data'/name
            archive.write(path,path.relative_to(BASE_DIR))
    from google.colab import files
    files.download(str(target))
else:
    print('All pipeline outputs and evidence saved. Optional Colab download is off.')

All pipeline outputs and evidence saved. Optional Colab download is off.
